# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data dasar. Sesuaikan nama kolom dan aturan validasi dengan dataset masing-masing.

In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'dataset.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Pastikan Milestone 1 sudah menempatkan data di data/raw/.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [ ]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [ ]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

## 3. Penanganan Missing Values

Ganti `kolom_numerik` dan `kolom_kategorik` dengan nama kolom asli pada dataset. Pilih strategi sesuai jenis data: median/mean untuk numerik, nilai default atau modus untuk kategorik.

In [ ]:
# TODO: sesuaikan nama kolom dengan dataset Anda
df_missing_handled = df.with_columns([
    # Contoh: isi kolom numerik dengan median
    # pl.col('kolom_numerik').fill_null(pl.col('kolom_numerik').median()),

    # Contoh: isi kolom kategorik dengan nilai default
    # pl.col('kolom_kategorik').fill_null('tidak diketahui'),
])

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')

## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis (domain knowledge) yang relevan dengan dataset Anda.

In [ ]:
# TODO: ganti 'kolom_numerik' dengan kolom numerik utama pada dataset Anda
KOLOM_NUMERIK = 'kolom_numerik'

q1 = df_missing_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_missing_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1}, Q3={q3}, IQR={iqr}')
print(f'Batas bawah: {batas_bawah}, Batas atas: {batas_atas}')

# Windsorization: clip nilai ke batas yang valid, bukan menghapus barisnya
df_outlier_handled = df_missing_handled.with_columns(
    pl.col(KOLOM_NUMERIK).clip(lower_bound=batas_bawah, upper_bound=batas_atas)
)

## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

In [ ]:
# TODO: ganti 'id_unik' dengan kolom kunci unik pada dataset Anda
KOLOM_KUNCI = 'id_unik'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

## 6. Validasi Data Dasar

Tulis beberapa aturan validasi sederhana untuk memastikan data bersih memenuhi batas yang masuk akal. Gantikan contoh aturan di bawah dengan aturan yang relevan dengan dataset Anda.

In [ ]:
connection.register('dataset_bersih', df_dedup)

# TODO: sesuaikan aturan validasi dengan kolom dan batas nilai pada dataset Anda
validasi = connection.execute(f"""
    SELECT
        COUNT(*) AS total_baris,
        SUM(CASE WHEN {KOLOM_NUMERIK} IS NULL THEN 1 ELSE 0 END) AS pelanggaran_null,
        SUM(CASE WHEN {KOLOM_NUMERIK} < {batas_bawah} OR {KOLOM_NUMERIK} > {batas_atas}
            THEN 1 ELSE 0 END) AS pelanggaran_batas
    FROM dataset_bersih
""").pl()
validasi

## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {len(df):,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')

In [2]:

# ============================================================
# TUGAS 1 ANALISIS BIG DATA 2026
# MILESTONE 2 - DATA CLEANING DAN DUCKDB PROFILING
# Dataset: Nationwide Agroclimatic Dataset for Indonesia
# ============================================================

# 1. IMPORT LIBRARY
from pathlib import Path
import polars as pl
import duckdb
import pandas as pd

print("Library berhasil diimpor!")

# ============================================================
# 2. MENENTUKAN LOKASI DATASET
# ============================================================

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "agroclimatic_indonesia"
    / "data"
    / "processed"
    / "agroclimate_with_indicators.parquet"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_PATH = OUTPUT_DIR / "agroclimate_bersih.parquet"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nLokasi dataset:", DATA_PATH)
print("Dataset ditemukan:", DATA_PATH.exists())

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

# ============================================================
# 3. MEMBACA DATASET MENGGUNAKAN POLARS
# ============================================================

df = pl.read_parquet(DATA_PATH)

print("\n========== INFORMASI DATASET AWAL ==========")
print("Jumlah baris:", df.height)
print("Jumlah kolom:", df.width)
print("Ukuran kolom:", df.columns)

display(df.head())

# ============================================================
# 4. PROFILING MENGGUNAKAN DUCKDB SUMMARIZE
# ============================================================

connection = duckdb.connect()
connection.register("dataset", df)

profil = connection.execute(
    "SUMMARIZE SELECT * FROM dataset"
).fetchdf()

print("\n========== DUCKDB PROFILING DATASET AWAL ==========")
display(profil)

# ============================================================
# 5. MEMERIKSA TIPE DATA
# ============================================================

print("\n========== TIPE DATA ==========")

for kolom, tipe in zip(df.columns, df.dtypes):
    print(f"{kolom}: {tipe}")

# ============================================================
# 6. MEMERIKSA MISSING VALUES
# ============================================================

jumlah_baris = df.height

null_summary = pl.DataFrame({
    "kolom": df.columns,
    "jumlah_null": [
        df[col].null_count()
        for col in df.columns
    ]
}).with_columns(
    (
        pl.col("jumlah_null") / jumlah_baris * 100
    ).round(2).alias("persen_null")
).sort("jumlah_null", descending=True)

print("\n========== MISSING VALUES DATASET AWAL ==========")
display(null_summary)

# ============================================================
# 7. MEMERIKSA DUPLIKAT BERDASARKAN GRID_ID DAN DATE
# ============================================================

duplikat = (
    df.group_by(["grid_id", "date"])
    .len()
    .filter(pl.col("len") > 1)
)

print("\n========== PEMERIKSAAN DUPLIKAT ==========")
print("Jumlah kelompok kunci duplikat:", duplikat.height)

if duplikat.height > 0:
    display(duplikat.head(10))
else:
    print("Tidak ditemukan duplikat berdasarkan grid_id dan date.")

# ============================================================
# 8. MEMERIKSA QUALITY CONTROL
# ============================================================

qc_summary = (
    df.group_by("qc_range_flag")
    .len()
    .sort("qc_range_flag")
)

print("\n========== DISTRIBUSI QC RANGE FLAG ==========")
display(qc_summary)

# ============================================================
# 9. STATISTIK DESKRIPTIF DATA IKLIM
# ============================================================

kolom_iklim = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "relative_humidity_2m_mean",
    "et0_fao_evapotranspiration"
]

print("\n========== STATISTIK DESKRIPTIF ==========")
display(df.select(kolom_iklim).describe())

# ============================================================
# 10. DATA CLEANING MENGGUNAKAN POLARS
# ============================================================

jumlah_awal = df.height

# Menghapus duplikat pada seluruh kolom
df_clean = df.unique()

# Menghapus baris tanpa grid_id dan date
df_clean = df_clean.drop_nulls(
    subset=["grid_id", "date"]
)

jumlah_akhir = df_clean.height
jumlah_dihapus = jumlah_awal - jumlah_akhir

print("\n========== HASIL DATA CLEANING ==========")
print("Jumlah baris sebelum cleaning:", jumlah_awal)
print("Jumlah baris setelah cleaning:", jumlah_akhir)
print("Jumlah baris yang dihapus:", jumlah_dihapus)

# ============================================================
# 11. VALIDASI DATA SETELAH CLEANING MENGGUNAKAN DUCKDB
# ============================================================

connection.register("dataset_bersih", df_clean)

validasi = connection.execute(
    "SUMMARIZE SELECT * FROM dataset_bersih"
).fetchdf()

print("\n========== DUCKDB PROFILING SETELAH CLEANING ==========")
display(validasi)

# ============================================================
# 12. MEMERIKSA MISSING VALUES SETELAH CLEANING
# ============================================================

null_setelah = pl.DataFrame({
    "kolom": df_clean.columns,
    "jumlah_null": [
        df_clean[col].null_count()
        for col in df_clean.columns
    ]
}).with_columns(
    (
        pl.col("jumlah_null") / df_clean.height * 100
    ).round(2).alias("persen_null")
).sort("jumlah_null", descending=True)

print("\n========== MISSING VALUES SETELAH CLEANING ==========")
display(null_setelah)

# ============================================================
# 13. MENYIMPAN DATASET HASIL CLEANING
# ============================================================

df_clean.write_parquet(OUTPUT_PATH)

print("\n========== PENYIMPANAN DATASET ==========")
print("Dataset berhasil disimpan!")
print("Lokasi:", OUTPUT_PATH)
print("Jumlah baris:", df_clean.height)
print("Jumlah kolom:", df_clean.width)

# ============================================================
# 14. MEMBACA ULANG DAN MEMVERIFIKASI DATASET
# ============================================================

df_final = pl.read_parquet(OUTPUT_PATH)

print("\n========== VERIFIKASI DATASET AKHIR ==========")
print("Jumlah baris:", df_final.height)
print("Jumlah kolom:", df_final.width)
print(
    "Ukuran file:",
    round(OUTPUT_PATH.stat().st_size / (1024**2), 2),
    "MB"
)

display(df_final.head())

# ============================================================
# 15. KESIMPULAN SEMENTARA
# ============================================================

print("\n========== KESIMPULAN MILESTONE 2 ==========")
print("Dataset berhasil dibaca menggunakan Polars.")
print("Profiling dataset dilakukan menggunakan DuckDB SUMMARIZE.")
print("Missing values dan duplikat telah diperiksa.")
print("Data cleaning dilakukan menggunakan Polars.")
print("Dataset hasil cleaning berhasil disimpan.")
print("\nOUTPUT:", OUTPUT_PATH)

connection.close()


Library berhasil diimpor!

Lokasi dataset: C:\Users\ADMIN\OneDrive\Documents\tugas1-abd-2026\data\raw\agroclimatic_indonesia\data\processed\agroclimate_with_indicators.parquet
Dataset ditemukan: True

========== INFORMASI DATASET AWAL ==========
Jumlah baris: 2235636
Jumlah kolom: 23
Ukuran kolom: ['grid_id', 'latitude', 'longitude', 'date', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'relative_humidity_2m_mean', 'et0_fao_evapotranspiration', 'qc_range_flag', 'rainfall_7d', 'rainfall_30d', 'rainfall_90d', 'rainy_days_30d', 'rainy_days_90d', 'max_daily_rainfall_30d', 'consecutive_dry_days', 'temperature_range', 'rolling_temperature_mean_7d', 'cumulative_et0_30d', 'water_balance', 'rolling_water_balance_30d']


grid_id,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean,et0_fao_evapotranspiration,qc_range_flag,rainfall_7d,rainfall_30d,rainfall_90d,rainy_days_30d,rainy_days_90d,max_daily_rainfall_30d,consecutive_dry_days,temperature_range,rolling_temperature_mean_7d,cumulative_et0_30d,water_balance,rolling_water_balance_30d
str,f64,f64,datetime[ns],f64,f64,f64,f64,i64,f64,bool,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64
"""GRID_+00000_+100000""",0.0,100.0,2016-01-01 00:00:00,23.1,24.1,21.9,28.0,95,1.11,false,28.0,28.0,28.0,1.0,1.0,28.0,0,2.2,23.1,1.11,26.89,26.89
"""GRID_+00000_+100000""",0.0,100.0,2016-01-02 00:00:00,23.1,26.5,21.4,14.2,93,2.35,false,42.2,42.2,42.2,2.0,2.0,28.0,0,5.1,23.1,3.46,11.85,38.74
"""GRID_+00000_+100000""",0.0,100.0,2016-01-03 00:00:00,24.2,29.8,20.0,1.9,88,4.3,false,44.1,44.1,44.1,3.0,3.0,28.0,0,9.8,23.466667,7.76,-2.4,36.34
"""GRID_+00000_+100000""",0.0,100.0,2016-01-04 00:00:00,24.0,28.2,21.6,8.6,91,3.55,false,52.7,52.7,52.7,4.0,4.0,28.0,0,6.6,23.6,11.31,5.05,41.39
"""GRID_+00000_+100000""",0.0,100.0,2016-01-05 00:00:00,24.0,28.5,21.7,4.7,91,3.65,false,57.4,57.4,57.4,5.0,5.0,28.0,0,6.8,23.68,14.96,1.05,42.44



========== DUCKDB PROFILING DATASET AWAL ==========


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,grid_id,VARCHAR,GRID_+00000_+100000,GRID_-10000_+124500,604,NaN,NaN,NaN,NaN,NaN,2235636,0.0
1,latitude,DOUBLE,-10.0,5.5,45,-2.1738562099673224,3.301001190380086,-4.0,-2.0,0.31851980417996895,2235636,0.0
2,longitude,DOUBLE,95.5,140.5,109,116.99019608823433,12.789912886375356,106.16984002407699,115.26294854688295,124.97857199384259,2235636,0.0
3,date,TIMESTAMP_NS,2016-01-01 00:00:00,2025-12-31 00:00:00,3794,2020-12-31 00:00:00,NaN,2018-07-02 07:03:23.096189,2020-12-27 18:38:22.100647,2023-07-02 10:31:22.602215,2235636,0.0
4,temperature_2m_mean,DOUBLE,4.0,32.1,251,24.793701568591665,2.78245209775979,23.973524251186518,25.558453850876962,26.511183381801647,2235636,0.0
5,temperature_2m_max,DOUBLE,5.3,47.5,354,28.355297552911143,2.9713993372656153,27.20664159491842,28.903509745276857,30.244857216023554,2235636,0.0
6,temperature_2m_min,DOUBLE,0.1,28.8,281,22.334884390839967,2.85636889711279,21.441829127575044,23.12887744893267,24.023596344007778,2235636,0.0
7,precipitation_sum,DOUBLE,0.0,460.0,2237,9.234793410018394,12.437869341472734,1.6063957704391403,5.996129432617184,12.72201985477394,2235636,0.0
8,relative_humidity_2m_mean,BIGINT,32,100,64,87.27345015020335,6.156057046463617,84,88,91,2235636,0.0
9,et0_fao_evapotranspiration,DOUBLE,0.26,8.18,732,3.4442627780193393,0.9366539930660723,2.8514997914806495,3.505943532152576,4.078005451024973,2235636,0.0



========== TIPE DATA ==========
grid_id: String
latitude: Float64
longitude: Float64
date: Datetime(time_unit='ns', time_zone=None)
temperature_2m_mean: Float64
temperature_2m_max: Float64
temperature_2m_min: Float64
precipitation_sum: Float64
relative_humidity_2m_mean: Int64
et0_fao_evapotranspiration: Float64
qc_range_flag: Boolean
rainfall_7d: Float64
rainfall_30d: Float64
rainfall_90d: Float64
rainy_days_30d: Float64
rainy_days_90d: Float64
max_daily_rainfall_30d: Float64
consecutive_dry_days: Int64
temperature_range: Float64
rolling_temperature_mean_7d: Float64
cumulative_et0_30d: Float64
water_balance: Float64
rolling_water_balance_30d: Float64

========== MISSING VALUES DATASET AWAL ==========


kolom,jumlah_null,persen_null
str,i64,f64
"""grid_id""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0
"""date""",0,0.0
"""temperature_2m_mean""",0,0.0
…,…,…
"""temperature_range""",0,0.0
"""rolling_temperature_mean_7d""",0,0.0
"""cumulative_et0_30d""",0,0.0


... (7 outputs left out)

kolom,jumlah_null,persen_null
str,i64,f64
"""grid_id""",0,0.0
"""latitude""",0,0.0
"""longitude""",0,0.0
"""date""",0,0.0
"""temperature_2m_mean""",0,0.0
…,…,…
"""temperature_range""",0,0.0
"""rolling_temperature_mean_7d""",0,0.0
"""cumulative_et0_30d""",0,0.0



========== PENYIMPANAN DATASET ==========
Dataset berhasil disimpan!
Lokasi: C:\Users\ADMIN\OneDrive\Documents\tugas1-abd-2026\data\processed\agroclimate_bersih.parquet
Jumlah baris: 2235636
Jumlah kolom: 23

========== VERIFIKASI DATASET AKHIR ==========
Jumlah baris: 2235636
Jumlah kolom: 23
Ukuran file: 90.26 MB


grid_id,latitude,longitude,date,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,relative_humidity_2m_mean,et0_fao_evapotranspiration,qc_range_flag,rainfall_7d,rainfall_30d,rainfall_90d,rainy_days_30d,rainy_days_90d,max_daily_rainfall_30d,consecutive_dry_days,temperature_range,rolling_temperature_mean_7d,cumulative_et0_30d,water_balance,rolling_water_balance_30d
str,f64,f64,datetime[ns],f64,f64,f64,f64,i64,f64,bool,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64
"""GRID_-09500_+119500""",-9.5,119.5,2021-03-21 00:00:00,24.0,26.3,22.3,7.7,91,3.12,false,64.3,308.2,1016.1,29.0,84.0,22.3,0,4.0,24.171429,99.53,4.58,208.67
"""GRID_+01500_+099500""",1.5,99.5,2021-06-20 00:00:00,24.5,28.8,21.4,1.7,84,3.45,false,19.3,91.3,513.6,16.0,73.0,17.4,0,7.4,25.057143,109.34,-1.75,-18.04
"""GRID_-05500_+138500""",-5.5,138.5,2025-05-22 00:00:00,25.0,27.5,23.7,11.2,95,1.3,false,106.9,329.8,1017.1,24.0,77.0,73.0,0,3.8,25.785714,81.9,9.9,247.9
"""GRID_-03500_+140500""",-3.5,140.5,2017-03-12 00:00:00,24.4,28.2,22.6,2.1,89,1.94,false,59.7,219.7,766.4,29.0,82.0,18.7,0,5.6,24.9,102.82,0.16,116.88
"""GRID_-02500_+116000""",-2.5,116.0,2017-05-15 00:00:00,25.9,29.5,23.2,4.3,89,3.27,false,54.0,301.3,1047.7,29.0,89.0,32.1,0,6.3,25.828571,94.65,1.03,206.65



========== KESIMPULAN MILESTONE 2 ==========
Dataset berhasil dibaca menggunakan Polars.
Profiling dataset dilakukan menggunakan DuckDB SUMMARIZE.
Missing values dan duplikat telah diperiksa.
Data cleaning dilakukan menggunakan Polars.
Dataset hasil cleaning berhasil disimpan.

OUTPUT: C:\Users\ADMIN\OneDrive\Documents\tugas1-abd-2026\data\processed\agroclimate_bersih.parquet


## Catatan Milestone 2

Dokumentasikan di sini: dimensi kualitas data yang paling bermasalah, strategi yang dipilih untuk missing values dan outlier, jumlah baris yang berubah, serta aturan validasi yang diterapkan.